In [1]:
# FAAC Pilot A: train light heads on cached frozen features (no image reading, no backbone).
# Inputs to attach in a NEW Kaggle notebook (GPU T4 x1 is enough; CPU also works, slower):
#   1. competition data: rsna-knee-abnormality-detection   (train_series.csv, train.csv)
#   2. dataset: tascj0/rsna-2026-knee-agent-labels           (Astra soft labels)
#   3. the committed extraction notebook's output             (feats/*.npz, scanner_split.csv)
#
# Arms (same features, labels, model, optimiser, seeds; only the training objective differs):
#   A        global pooling, no consistency
#   B        anatomy-box pooling, no consistency
#   Uniform  B + consistency between ALL series pairs of a study
#   Random   B + consistency on the same NUMBER of pairs as FAAC, chosen at random each step
#   FAAC     B + consistency only on medically comparable pairs (v1 rules)
# Consistency is applied to projected box features (cosine distance) for the focal findings only.
# Evaluation: per-finding ROC-AUC against Astra labels (>= 0.5) on val_id, test_GE, test_Canon.
import os, glob, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

# QUICK = True: logic test on ~600 studies, 1 seed, 3 epochs, with PASS/FAIL checks (a few minutes).
# Run it interactively first. If every check passes, set QUICK = False and Commit the full run.
QUICK = True

ARMS = ["A", "B", "Uniform", "Random", "FAAC"]
SEEDS = [0] if QUICK else list(range(10))
EPOCHS, BATCH, LR, WD, LAMBDA, DIM = (3 if QUICK else 25), 128, 1e-3, 1e-2, 0.1, 256
FINDINGS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
            "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
FOCAL = ["Contusion", "Effusion", "Medial Meniscus", "Lateral Meniscus", "Fracture"]
MACRO = [f for f in FINDINGS if f != "MCL"]          # MCL has too few positives on held-out vendors
SPLITS = ["val_id", "test_GE", "test_Canon"]
OUT = "/kaggle/working/pilot_a"
os.makedirs(OUT, exist_ok=True)
dev = "cuda" if torch.cuda.is_available() else "cpu"

def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"not found under /kaggle/input: {pattern}"
    return hits[0]

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
FEAT_DIR = os.path.dirname(find("feats/*.npz"))
series = pd.read_csv(os.path.join(COMP, "train_series.csv")).set_index("SeriesInstanceUID")
split = pd.read_csv(find("scanner_split.csv")).set_index("StudyInstanceUID")
lab = pd.read_csv(find("labels_astra_trained_guide.csv"))
lab = lab.set_index(next(c for c in lab.columns if "study" in c.lower()))[FINDINGS]

# the 58 expert-labelled studies are reserved for the final check: exclude them here
tr = pd.read_csv(os.path.join(COMP, "train.csv"))
tr_id = next(c for c in tr.columns if "study" in c.lower())
lab_cols = [c for c in FINDINGS if c in tr.columns]                  # only the 12 label columns
print("train.csv columns:", list(tr.columns)[:16])
gold = set(tr.loc[tr[lab_cols].notna().any(axis=1), tr_id].astype(str)) if lab_cols else set()
print(f"features: {FEAT_DIR}\ngold studies excluded: {len(gold)}")

# ---------- load features into padded tensors ----------
files = sorted(glob.glob(f"{FEAT_DIR}/*.npz"))
print(f"feature files: {len(files)}")
if QUICK:
    files = list(np.random.RandomState(0).choice(files, min(700, len(files)), replace=False))
studies, feats, meta = [], [], []
for f in files:
    sid = os.path.basename(f)[:-4]
    if sid in gold or sid not in split.index or sid not in lab.index:
        continue
    if split.loc[sid, "split"] not in ["train"] + SPLITS:
        continue
    z = np.load(f)
    sers = [s for s in z.files if s in series.index]
    if not sers:
        continue
    studies.append(sid)
    feats.append(np.stack([z[s] for s in sers]))                       # [S, 13, 768]
    meta.append(series.loc[sers, ["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]].values)
N, S = len(studies), max(len(x) for x in feats)
X = np.zeros((N, S, 13, 768), np.float16)
M = np.zeros((N, S), bool)
for i, x in enumerate(feats):
    X[i, :len(x)] = x
    M[i, :len(x)] = True
Y = lab.loc[studies].values.astype(np.float32)                         # Astra soft labels
SPL = split.loc[studies, "split"].values
print(f"studies: {N}, max series per study: {S}")
print(pd.Series(SPL).value_counts().to_string())

# ---------- integrity check: all-zero ROI features mean "no box found" and must be masked ----------
V = np.abs(X[:, :, :12]).max(-1) > 0                                   # [N, S, 12] valid ROI feature
V &= M[:, :, None]
vendor_of = split.loc[studies, "vendor"].values if "vendor" in split.columns else SPL
zero = pd.DataFrame(~V[M].reshape(-1, 12), columns=FINDINGS)
zero["group"] = np.repeat(vendor_of, M.sum(1))
print("\n== fraction of series with an all-zero ROI feature, by vendor (should be ~0)")
print(zero.groupby("group").mean().round(4).to_string())
print("all-zero global features:", int((np.abs(X[:, :, 12]).max(-1) == 0)[M].sum()))

# ---------- comparable-pair masks (v1 rules) ----------
K = len(FOCAL)
uniform = np.zeros((N, K, S, S), bool)   # per finding: all pairs whose ROIs are both valid
faac = np.zeros((N, K, S, S), bool)
for n, m in enumerate(meta):
    plane, fluid, fs = m[:, 0], m[:, 1].astype(int), m[:, 2].astype(int)
    for i in range(len(m)):
        for j in range(i + 1, len(m)):
            fluid_fs = fluid[i] == 1 and fluid[j] == 1 and fs[i] == 1 and fs[j] == 1
            same_plane = plane[i] == plane[j]
            struct_pair = fluid[i] == 0 and fluid[j] == 0 and same_plane
            for k, f in enumerate(FOCAL):
                kk = FINDINGS.index(f)
                if not (V[n, i, kk] and V[n, j, kk]):                  # never pair a missing ROI (all arms)
                    continue
                uniform[n, k, i, j] = True
                if f in ("Contusion", "Effusion"):
                    ok = fluid_fs
                elif f in ("Medial Meniscus", "Lateral Meniscus"):
                    ok = same_plane and plane[i] in ("Sagittal", "Coronal")
                elif f == "Fracture":
                    ok = fluid_fs or struct_pair
                faac[n, k, i, j] = ok
n_faac = faac.sum((2, 3))                                              # [N, K] pairs per study
print("mean pairs per study: uniform %s | FAAC %s" % (dict(zip(FOCAL, uniform.sum((2, 3)).mean(0).round(1).tolist())),
      dict(zip(FOCAL, n_faac.mean(0).round(1).tolist()))))

Xt = torch.from_numpy(X).to(dev)
Mt, Yt = torch.from_numpy(M).to(dev), torch.from_numpy(Y).to(dev)
Vt = torch.from_numpy(V).to(dev)
Ut, Ft, NFt = torch.from_numpy(uniform).to(dev), torch.from_numpy(faac).to(dev), torch.from_numpy(n_faac).to(dev)
focal_idx = torch.tensor([FINDINGS.index(f) for f in FOCAL], device=dev)
train_idx = np.where(SPL == "train")[0]

# ---------- model ----------
class Head(nn.Module):
    def __init__(self, use_boxes):
        super().__init__()
        self.use_boxes = use_boxes
        self.proj = nn.Sequential(nn.Dropout(0.1), nn.Linear(768, DIM), nn.GELU(), nn.LayerNorm(DIM))
        self.W = nn.Parameter(torch.randn(12, DIM) * 0.02)
        self.b = nn.Parameter(torch.zeros(12))
    def forward(self, x, m, v):
        x = x.float()                                                   # [B, S, 13, 768]
        if self.use_boxes:
            x, w = x[:, :, :12], (m[:, :, None] & v).float()[..., None]  # skip series with no ROI
        else:
            x, w = x[:, :, 12:13].expand(-1, -1, 12, -1), m[:, :, None, None].float().expand(-1, -1, 12, -1)
        z = self.proj(x)                                                # [B, S, 12, DIM]
        rep = (z * w).sum(1) / w.sum(1).clamp(min=1)                    # masked mean over series
        return (rep * self.W).sum(-1) + self.b, z

def consistency(z, pairs):
    """z: [B, S, 12, D]; pairs: [B, K, S, S] bool. Mean cosine distance over selected pairs."""
    zf = F.normalize(z[:, :, focal_idx], dim=-1).permute(0, 2, 1, 3)    # [B, K, S, D]
    d = 1 - zf @ zf.transpose(-1, -2)                                   # [B, K, S, S]
    p = pairs.float()
    return (d * p).sum() / p.sum().clamp(min=1)

def random_pairs(idx):
    """Same number of pairs per (study, finding) as FAAC, drawn at random from all pairs."""
    u = Ut[idx]                                                         # [B, K, S, S] valid candidate pairs
    score = torch.rand(u.shape, device=dev).masked_fill(~u, -1.0).flatten(2)
    rank = score.argsort(-1, descending=True).argsort(-1)
    return (rank < NFt[idx][..., None]).view(u.shape) & u

def train(arm, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = Head(use_boxes=(arm != "A")).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    rng = np.random.RandomState(seed)
    hist = []                                                           # per epoch: (cls loss, consistency)
    for ep in range(EPOCHS):
        model.train()
        cl, cc, nb = 0.0, 0.0, 0
        for b in np.array_split(rng.permutation(train_idx), max(1, len(train_idx) // BATCH)):
            idx = torch.from_numpy(b).to(dev)
            logits, z = model(Xt[idx], Mt[idx], Vt[idx])
            cls = F.binary_cross_entropy_with_logits(logits, Yt[idx])
            cons = torch.zeros((), device=dev)
            if arm == "Uniform":
                cons = consistency(z, Ut[idx])
            elif arm == "Random":
                cons = consistency(z, random_pairs(idx))
            elif arm == "FAAC":
                cons = consistency(z, Ft[idx])
            loss = cls + LAMBDA * cons
            opt.zero_grad(); loss.backward(); opt.step()
            cl, cc, nb = cl + cls.item(), cc + cons.item(), nb + 1
        hist.append((cl / nb, cc / nb))
    model.eval()
    with torch.no_grad():
        preds = torch.cat([torch.sigmoid(model(Xt[i:i + 256], Mt[i:i + 256], Vt[i:i + 256])[0]) for i in range(0, N, 256)])
    return preds.cpu().numpy(), hist, model

# ---------- logic checks (cheap; always run, printed as PASS / FAIL) ----------
def check(name, ok, detail=""):
    print(f"[{'PASS' if ok else 'FAIL'}] {name} {detail}")
    return ok

print("\n== logic checks")
check("58 expert studies found and excluded", len(gold) == 58, f"(found {len(gold)})")
check("no gold study in the loaded set", not set(studies) & gold)
check("features finite", bool(np.isfinite(X[M].astype(np.float32)).all()))
check("labels in [0, 1]", bool(((Y >= 0) & (Y <= 1)).all()))
check("every study has >= 1 series", bool(M.sum(1).min() >= 1), f"(mean {M.sum(1).mean():.1f})")
if "vendor" in split.columns:
    ven = split.loc[studies, "vendor"].values
    check("train/val_id only Siemens + Philips", set(ven[np.isin(SPL, ["train", "val_id"])]) <= {"Siemens", "Philips"})
    check("test_GE only GE", set(ven[SPL == "test_GE"]) <= {"GE"})
    check("test_Canon only Canon/Toshiba", set(ven[SPL == "test_Canon"]) <= {"Canon/Toshiba"})
check("FAAC pairs are a subset of uniform pairs", not (faac & ~uniform).any())
check("FAAC selects fewer pairs than uniform", faac.sum() < uniform.sum(),
      f"(FAAC {faac.sum()} vs uniform {uniform.sum()})")
# rule spot check on Contusion: every FAAC pair must be fluid-sensitive + fat-suppressed on both sides
kc, bad = FOCAL.index("Contusion"), 0
for n in range(min(N, 300)):
    m = meta[n]
    for i, j in zip(*np.where(faac[n, kc])):
        bad += not (m[i, 1] == 1 and m[j, 1] == 1 and m[i, 2] == 1 and m[j, 2] == 1)
check("Contusion pairs are all fluid + fat-suppressed", bad == 0, f"({bad} violations)")
idx = torch.arange(min(N, 256), device=dev)
rp = random_pairs(idx)
check("Random pairs: same count as FAAC per study and finding", bool((rp.sum((2, 3)) == NFt[idx]).all()))
check("Random pairs: subset of the same valid candidate pairs", not (rp & ~Ut[idx]).any())
overlap = (rp & Ft[idx]).sum().item() / max(Ft[idx].sum().item(), 1)
check("Random pairs differ from FAAC pairs", overlap < 0.95, f"(overlap {overlap:.2f})")
with torch.no_grad():
    mA, mB = Head(False).to(dev).eval(), Head(True).to(dev).eval()
    x = Xt[idx].clone(); x2 = x.clone(); x2[:, :, :12] += 1.0       # change ROI features only
    check("A ignores ROI features", torch.allclose(mA(x, Mt[idx], Vt[idx])[0], mA(x2, Mt[idx], Vt[idx])[0]))
    check("B uses ROI features", not torch.allclose(mB(x, Mt[idx], Vt[idx])[0], mB(x2, Mt[idx], Vt[idx])[0]))
    x3 = x.clone(); x3[:, :, 12] += 1.0                              # change global feature only
    check("B ignores the global feature", torch.allclose(mB(x, Mt[idx], Vt[idx])[0], mB(x3, Mt[idx], Vt[idx])[0]))
    zz = torch.ones(2, 3, 12, DIM, device=dev)
    check("consistency of identical features is 0", consistency(zz, torch.ones(2, K, 3, 3, dtype=torch.bool, device=dev)).item() < 1e-5)

def evaluate(pred, arm, seed):
    rows = []
    for sp in SPLITS:
        sel = SPL == sp
        y = (Y[sel] >= 0.5).astype(int)
        for k, f in enumerate(FINDINGS):
            if 10 <= y[:, k].sum() <= len(y) - 10:
                rows.append(dict(arm=arm, seed=seed, split=sp, finding=f,
                                 auc=roc_auc_score(y[:, k], pred[sel, k]), n_pos=int(y[:, k].sum())))
    return rows

# ---------- run ----------
rows, hist_rows, t0 = [], [], time.time()
for seed in SEEDS:
    for arm in ARMS:
        ta = time.time()
        pred, hist, _ = train(arm, seed)
        hist_rows += [dict(arm=arm, seed=seed, epoch=e, cls_loss=c, consistency=k) for e, (c, k) in enumerate(hist)]
        rows += evaluate(pred, arm, seed)
        np.save(f"{OUT}/pred_{arm}_seed{seed}.npy", pred.astype(np.float16))
        if seed == SEEDS[0]:
            (c0, k0), (c1, k1) = hist[0], hist[-1]
            print(f"  {arm:8s} cls loss {c0:.4f} -> {c1:.4f} | consistency {k0:.4f} -> {k1:.4f} | {time.time() - ta:.0f} s")
            check(f"{arm}: classification loss decreases", c1 < c0)
            if arm in ("Uniform", "Random", "FAAC"):
                check(f"{arm}: consistency term is active", k0 > 0)
                print(f"[INFO] {arm}: consistency {'decreased' if k1 < k0 else 'did not decrease'} "
                      f"over {EPOCHS} epochs (not required in QUICK mode; Random resamples pairs every step)")
            check(f"{arm}: predictions finite and not constant", bool(np.isfinite(pred).all() and pred.std(0).min() > 1e-4))
    print(f"seed {seed} done ({time.time() - t0:.0f} s)", flush=True)
if QUICK:
    print(f"\nQUICK run finished in {time.time() - t0:.0f} s. If all checks PASS, set QUICK = False and Commit."
          f"\nFull run estimate: about {(time.time() - t0) / len(ARMS) * len(ARMS) * 10 * (25 / 3) * (4400 / N):.0f} s "
          f"of training (rough, linear in seeds, epochs and studies).")
res = pd.DataFrame(rows)
res.to_csv(f"{OUT}/results_long.csv", index=False)
pd.DataFrame(hist_rows).to_csv(f"{OUT}/loss_history.csv", index=False)   # per arm, seed and epoch
pd.Series(studies).to_csv(f"{OUT}/study_order.csv", index=False)

# ---------- summaries ----------
pd.set_option("display.width", 250)
def macro(df, fs):
    return df[df.finding.isin(fs)].groupby(["arm", "seed", "split"]).auc.mean().unstack("split")

for name, fs in [("macro (11 findings, no MCL)", MACRO), ("focal findings", FOCAL)]:
    m = macro(res, fs)
    print(f"\n== {name}: mean ± std over {len(SEEDS)} seeds")
    print(m.groupby("arm").agg(["mean", "std"]).reindex(ARMS).round(4).to_string())

print("\n== per-finding AUC, mean over seeds")
print(res.groupby(["split", "finding", "arm"]).auc.mean().unstack("arm")[ARMS].round(3).to_string())

# paired deltas: same seed => same initialisation and batch order across arms
print("\n== FAAC minus other arms, paired by seed (focal findings): mean, std, #seeds FAAC better")
mf = macro(res, FOCAL)
for other in ["B", "Uniform", "Random"]:
    d = mf.xs("FAAC", level="arm") - mf.xs(other, level="arm")
    print(f"vs {other:8s}", {sp: f"{d[sp].mean():+.4f} ± {d[sp].std():.4f} ({(d[sp] > 0).sum()}/{len(d)})"
                              for sp in SPLITS})

print("\n== sanity check: B minus A on meniscus (should be > 0 if boxes are aligned with slices)")
pf = res[res.finding.isin(["Medial Meniscus", "Lateral Meniscus"])].groupby(["arm", "split"]).auc.mean().unstack()
print((pf.loc["B"] - pf.loc["A"]).round(4).to_string())
print(f"\ntotal time {time.time() - t0:.0f} s; outputs in {OUT}")

train.csv columns: ['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
features: /kaggle/input/notebooks/juedu5/notebooka2d2d91b07/feats
gold studies excluded: 58
feature files: 4407
studies: 691, max series per study: 11
train         430
test_GE       151
val_id         77
test_Canon     33

== fraction of series with an all-zero ROI feature, by vendor (should be ~0)
               ACL  MCL  Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA  PF OA  Effusion  Synovitis  Baker's  Contusion  Fracture
group                                                                                                                                      
Canon/Toshiba  0.0  0.0              0.0               0.0        0.0         0.0    0.0       0.0        0.0      0.0        0.0       0.0
GE             0.0  0.0              0.0               0.0        0.0         0